# Imports

In [1]:
import os
import pickle
import subprocess
import tempfile
import xml.etree.ElementTree as ET
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import random

# XML Field helpers

In [2]:
def _safe_name(raw: Optional[str], empty_name: str) -> str:
    if raw is None or raw == "":
        return empty_name
    return raw.replace(".", "_")


def get_nested_field_names(field_element: ET.Element) -> List[str]:
    nested_field_names: List[str] = []
    for nested_field in field_element.findall("field"):
        field_name = _safe_name(nested_field.get("name"), "nested_empty_field_name")
        nested_field_names.extend([
            f"{field_name}_name",
            f"{field_name}_showname",
            f"{field_name}_size",
            f"{field_name}_pos",
            f"{field_name}_show",
            f"{field_name}_value",
        ])
        nested_field_names.extend(get_nested_field_names(nested_field))
    return nested_field_names


def get_packet_field_names(proto: ET.Element) -> List[str]:
    packet_fields: List[str] = []
    for field in proto.findall("field"):
        field_name = _safe_name(field.get("name"), "empty_field_name")
        packet_fields.extend([
            f"{field_name}_name",
            f"{field_name}_showname",
            f"{field_name}_size",
            f"{field_name}_pos",
            f"{field_name}_show",
            f"{field_name}_value",
        ])
        packet_fields.extend(get_nested_field_names(field))
    return packet_fields


def get_nested_fields(field_element: ET.Element) -> List[Optional[str]]:
    nested_fields: List[Optional[str]] = []
    for nested_field in field_element.findall("field"):
        nested_fields.extend([
            nested_field.get("name"),
            nested_field.get("showname"),
            nested_field.get("size"),
            nested_field.get("pos"),
            nested_field.get("show"),
            nested_field.get("value"),
        ])
        nested_fields.extend(get_nested_fields(nested_field))
    return nested_fields

# Extraction Core

In [3]:
def align_lists(target_cols: List[str], cols: List[str], values: List[Optional[str]]) -> List[Optional[str]]:
    first_idx: Dict[str, int] = {}
    for i, c in enumerate(cols):
        if c not in first_idx:
            first_idx[c] = i

    out: List[Optional[str]] = []
    for c in target_cols:
        out.append(values[first_idx[c]] if c in first_idx else "")
    return out


def iter_rrc_protos(packet: ET.Element):
    for proto in packet.findall("proto"):
        if proto.get("name") == "nr-rrc" and proto.get("hide") != "yes":
            yield proto

    for top_proto in packet.findall("proto"):
        if top_proto.get("name") == "mac-nr":
            for proto in top_proto.iter("proto"):
                if proto.get("name") == "nr-rrc" and proto.get("hide") != "yes":
                    yield proto


def extract_columns_and_values_from_xml(xml_file: str) -> Tuple[List[str], List[List[Optional[str]]]]:
    tree = ET.parse(xml_file)
    root = tree.getroot()
    packets = root.findall("packet")

    column_names: set[str] = set()
    for packet in packets:
        for rrc_proto in iter_rrc_protos(packet):
            column_names.update(get_packet_field_names(rrc_proto))
            for nas_proto in rrc_proto.iter("proto"):
                if nas_proto.get("name") == "nas-5gs":
                    column_names.update(get_packet_field_names(nas_proto))

    ordered_columns = sorted(column_names)

    rows: List[List[Optional[str]]] = []
    for packet in packets:
        for rrc_proto in iter_rrc_protos(packet):
            packet_fields: List[Optional[str]] = []
            cols: List[str] = []

            for field in rrc_proto.findall("field"):
                packet_fields.extend([
                    field.get("name"),
                    field.get("showname"),
                    field.get("size"),
                    field.get("pos"),
                    field.get("show"),
                    field.get("value"),
                ])
                packet_fields.extend(get_nested_fields(field))
            cols.extend(get_packet_field_names(rrc_proto))

            for nas_proto in rrc_proto.iter("proto"):
                if nas_proto.get("name") == "nas-5gs":
                    for field in nas_proto.findall("field"):
                        packet_fields.extend([
                            field.get("name"),
                            field.get("showname"),
                            field.get("size"),
                            field.get("pos"),
                            field.get("show"),
                            field.get("value"),
                        ])
                        packet_fields.extend(get_nested_fields(field))
                    cols.extend(get_packet_field_names(nas_proto))

            rows.append(align_lists(ordered_columns, cols, packet_fields))

    return ordered_columns, rows

# PCAP/XML conversion helpers

In [4]:
def run_tshark_to_pdml(input_pcap: str, xml_output_file: str) -> None:
    with open(xml_output_file, "w", encoding="utf-8") as f:
        subprocess.run(["tshark", "-r", input_pcap, "-T", "pdml"], stdout=f, check=True)


def prepare_dataframe_from_xml(xml_file: str) -> pd.DataFrame:
    columns, values = extract_columns_and_values_from_xml(xml_file)
    return pd.DataFrame(values, columns=columns)


def prepare_dataframe_from_pcap(pcap_file: str) -> pd.DataFrame:
    with tempfile.NamedTemporaryFile(suffix=".xml", delete=False) as tmp:
        xml_path = tmp.name
    try:
        run_tshark_to_pdml(pcap_file, xml_path)
        return prepare_dataframe_from_xml(xml_path)
    finally:
        if os.path.exists(xml_path):
            os.remove(xml_path)


def list_pcaps(folder: str, recursive: bool = False) -> List[str]:
    exts = (".pcap", ".pcapng", ".cap")
    out: List[str] = []
    if recursive:
        for root, _, files in os.walk(folder):
            for f in files:
                if f.lower().endswith(exts):
                    out.append(os.path.join(root, f))
    else:
        for f in os.listdir(folder):
            p = os.path.join(folder, f)
            if os.path.isfile(p) and f.lower().endswith(exts):
                out.append(p)
    return sorted(out)

# Dataset building + encoding

In [ ]:
def build_raw_dataset(
    benign_dir: str,
    attack_dir: str,
    recursive: bool = False,
    shuffle_pcaps: bool = False,
    seed: Optional[int] = 42,
) -> Tuple[pd.DataFrame, Dict[str, int]]:
    all_frames: List[pd.DataFrame] = []
    session_map: Dict[str, int] = {}
    seq = 1

    # Build one combined job list first
    pcap_jobs: List[Tuple[str, int, str]] = []
    for class_dir, class_label, class_name in [
        (benign_dir, 0, "benign"),
        (attack_dir, 1, "attack"),
    ]:
        for pcap_path in list_pcaps(class_dir, recursive=recursive):
            pcap_jobs.append((pcap_path, class_label, class_name))

    # Shuffle across both folders
    if shuffle_pcaps:
        rng = random.Random(seed)
        rng.shuffle(pcap_jobs)

    for pcap_path, class_label, class_name in pcap_jobs:
        session_name = os.path.splitext(os.path.basename(pcap_path))[0]
        session_map[session_name] = seq

        try:
            df_i = prepare_dataframe_from_pcap(pcap_path)
        except Exception as e:
            print(f"[WARN] Skipping {pcap_path}: {e}")
            seq += 1
            continue

        if df_i.empty:
            print(f"[WARN] Skipping empty extraction: {pcap_path}")
            seq += 1
            continue

        df_i["Sequence_Number"] = seq
        df_i["Session_Name"] = session_name
        df_i["label"] = class_label
        df_i["class_name"] = class_name
        all_frames.append(df_i)
        seq += 1

    if not all_frames:
        raise ValueError("No valid packets were extracted from the two folders.")

    df_raw = pd.concat(all_frames, axis=0, join="outer", ignore_index=True, sort=False)
    df_raw = df_raw.replace(r"^\s*$", np.nan, regex=True)
    df_raw["Sequence_Number"] = df_raw["Sequence_Number"].astype(int)
    df_raw["label"] = df_raw["label"].astype(int)
    return df_raw, session_map


def encode_dataset(df_raw: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Dict[str, int]]]:
    df = df_raw.copy()

    for c in ["Session_Name", "class_name"]:
        if c in df.columns:
            df = df.drop(columns=[c])

    protected = {"Sequence_Number", "label"}
    feature_cols = [c for c in df.columns if c not in protected]
    df[feature_cols] = df[feature_cols].replace(r"^\s*$", np.nan, regex=True)

    category_maps: Dict[str, Dict[str, int]] = {}

    # Special handling for FBS binary cause encoding
    special_col = "nas-5gs_mm_5gmm_cause_show"
    if special_col in df.columns:
        primary_causes = {7, 11, 15, 27}

        def encode_cause(val):
            try:
                cause = int(float(str(val)))
                if cause in primary_causes:
                    return 30
                elif cause > 0:
                    return 0
                else:
                    return -1
            except (ValueError, TypeError):
                return -1

        df[special_col] = df[special_col].apply(encode_cause).astype(np.int32)
        category_maps[special_col] = {
            "primary_fbs_cause": 30,
            "other_cause": 0,
            "no_cause": -1,
            "__type__": "fbs_binary",
            "__primary__": [7, 11, 15, 27],
        }

    for col in feature_cols:
        if col == special_col:
            continue
        s = df[col].astype("string")
        uniques = sorted([str(v) for v in s.dropna().unique().tolist()])
        mapping = {v: i for i, v in enumerate(uniques)}
        df[col] = s.map(mapping).fillna(-1).astype(np.int32)
        category_maps[col] = mapping

    ordered_cols = ["Sequence_Number", "label"] + [c for c in df.columns if c not in ["Sequence_Number", "label"]]
    df = df[ordered_cols]
    return df, category_maps

In [6]:
def drop_useless_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Drop byte position columns — irrelevant for classification
    pos_cols = [c for c in df.columns if c.endswith("_pos")]
    size_cols = [c for c in df.columns if c.endswith("_size")]
    name_cols = [c for c in df.columns if c.endswith("_name")]

    # Drop near-constant columns (same value in >95% of rows)
    nunique = df.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()

    # Drop columns with >80% missing (-1)
    missing_rate = (df == -1).mean()
    sparse_cols = missing_rate[missing_rate > 0.95].index.tolist()

    drop = set(pos_cols + size_cols + name_cols + constant_cols + sparse_cols)
    drop -= {"Sequence_Number", "label"}  # never drop these

    print(f"Dropping {len(drop)} useless columns, keeping {len(df.columns) - len(drop)}")
    return df.drop(columns=list(drop), errors="ignore") 

# Notebook runner function

In [7]:
def run_training_pipeline(
    benign_dir: str,
    attack_dir: str,
    raw_csv: str = "csv_files/train_raw.csv",
    encoded_csv: str = "csv_files/train_encoded.csv",
    artifacts_dir: str = "artifacts",
    session_map_csv: str = "artifacts/session_map.csv",
    recursive: bool = False,
    shuffle_pcaps: bool = False,
    seed: Optional[int] = 42,
):
    os.makedirs(os.path.dirname(raw_csv) or ".", exist_ok=True)
    os.makedirs(os.path.dirname(encoded_csv) or ".", exist_ok=True)
    os.makedirs(artifacts_dir, exist_ok=True)
    os.makedirs(os.path.dirname(session_map_csv) or ".", exist_ok=True)

    df_raw, session_map = build_raw_dataset(
        benign_dir, attack_dir, recursive=recursive, shuffle_pcaps=shuffle_pcaps, seed=seed
    )
    df_raw.to_csv(raw_csv, index=False)
    print(f"Saved raw CSV: {raw_csv} (rows={len(df_raw)}, cols={len(df_raw.columns)})")

    df_encoded, category_maps = encode_dataset(df_raw)
    df_encoded = drop_useless_columns(df_encoded)
    df_encoded.to_csv(encoded_csv, index=False)
    print(f"Saved encoded CSV: {encoded_csv} (rows={len(df_encoded)}, cols={len(df_encoded.columns)})")

    with open(os.path.join(artifacts_dir, "feature_encoding.pkl"), "wb") as f:
        pickle.dump(
            {
                "feature_cols": [c for c in df_encoded.columns if c not in ("Sequence_Number", "label")],
                "category_maps": category_maps,
            },
            f,
        )
    print(f"Saved encoding artifact: {os.path.join(artifacts_dir, 'feature_encoding.pkl')}")

    pd.DataFrame(
        [{"Session_Name": k, "Sequence_Number": v} for k, v in session_map.items()]
    ).sort_values("Sequence_Number").to_csv(session_map_csv, index=False)
    print(f"Saved session map: {session_map_csv}")

    return df_raw, df_encoded, session_map

In [8]:
BENIGN_DIR = "/home/brian/thesis_stuff/thesis_SNC/thesis_SNC/pcap_srsRAN_zmq/benign"
ATTACK_DIR = "/home/brian/thesis_stuff/thesis_SNC/thesis_SNC/pcap_srsRAN_zmq/attack"

df_raw, df_encoded, session_map = run_training_pipeline(
    benign_dir=BENIGN_DIR,
    attack_dir=ATTACK_DIR,
    raw_csv="csv_files_srsRAN_zmq/train_raw.csv",
    encoded_csv="csv_files_srsRAN_zmq/train_encoded.csv",
    artifacts_dir="artifacts_2",
    session_map_csv="artifacts_2/session_map.csv",
    recursive=True,
    shuffle_pcaps=True,
    seed=42,  # same seed => same shuffled order
)

df_raw.head(), df_encoded.head()

Saved raw CSV: csv_files_srsRAN_zmq/train_raw.csv (rows=6741, cols=2752)
Dropping 1858 useless columns, keeping 892
Saved encoded CSV: csv_files_srsRAN_zmq/train_encoded.csv (rows=6741, cols=892)
Saved encoding artifact: artifacts_2/feature_encoding.pkl
Saved session map: artifacts_2/session_map.csv


(  e212_mcc_name e212_mcc_pos e212_mcc_show  \
 0           NaN          NaN           NaN   
 1           NaN          NaN           NaN   
 2           NaN          NaN           NaN   
 3      e212.mcc            7           999   
 4           NaN          NaN           NaN   
 
                                   e212_mcc_showname e212_mcc_size  \
 0                                               NaN           NaN   
 1                                               NaN           NaN   
 2                                               NaN           NaN   
 3  Mobile Country Code (MCC): Private network (999)             2   
 4                                               NaN           NaN   
 
   e212_mcc_value e212_mnc_name e212_mnc_pos e212_mnc_show  \
 0            NaN           NaN          NaN           NaN   
 1            NaN           NaN          NaN           NaN   
 2            NaN           NaN          NaN           NaN   
 3           99f9      e212.mnc            8  